In [28]:
import os
import json
import time
import pyvisa as pv

In [8]:
# Check available VISA resources
rm = pv.ResourceManager()
print(rm.list_resources())

('USB0::0x1AB1::0x0588::DG1D120300062::INSTR', 'ASRL1::INSTR', 'ASRL4::INSTR')


In [9]:
rigol = rm.open_resource('USB0::0x1AB1::0x0588::DG1D120300062::INSTR') #  use the write and read methods (instead of query) with a little delay before the read
rigol.write('*IDN?')
time.sleep(0.01)
print(rigol.read())

RIGOL TECHNOLOGIES,DG1022 ,DG1D120300062,,00.02.00.06.00.02.05



In [26]:
# Define configuration file path
CONFIG_DIR = "config/rigol"
os.makedirs(CONFIG_DIR, exist_ok=True)

In [30]:
# Choose configuration file (or define new one)
config = "lock_shg.json"
config_file_path = os.path.join(CONFIG_DIR, config)

# Load configuration parameters
with open(config_file_path, 'r') as f:
    config_data = json.load(f)

### Define settings for CH1 and CH2

Either use the config file to load the parameters or define them manually below. You can also save into a file the parameters.

In [31]:
# Option 1: Set parameters directly from config_data
command1 = f"APPLy:{config_data['waveform1']} {config_data['frequency1']},{config_data['amplitude1']},{config_data['offset1']}"
command1_phase = f"PHAS {config_data['phase1']}"
print("Command for CH1:", command1, "Phase for CH1:", command1_phase)

command2 = f"APPLy:{config_data['waveform2']}:CH2 {config_data['frequency2']},{config_data['amplitude2']},{config_data['offset2']}"
command2_phase = f"PHAS:CH2 {config_data['phase2']}"
print("Command for CH2:", command2, "Phase for CH2:", command2_phase)

Command for CH1: APPLy:SIN 5000000.0,20.0,0.0 Phase for CH1: PHAS -70.0
Command for CH2: APPLy:SIN:CH2 5000000.0,2.0,0.0 Phase for CH2: PHAS:CH2 80.0


In [32]:
# Option 2: Set parameters directly in code
# CH1 settings
waveform1 = 'SIN'  # (SIN, SQU, RAMP, PULS, NOIS)
freq1 = 5e6        # Hz
amp1 = 20.0         # Vpp
offset1 = 0.0      # V
command1 = f'APPLy:{waveform1} {freq1},{amp1},{offset1}'

phase1 = -70.0         # deg
command1_phase = f'PHAS {phase1}'

# CH2 settings
waveform2 = 'SIN'  # (SIN, SQU, RAMP, PULS, NOIS)
freq2 = 5e6       # Hz
amp2 = 2.0        # Vpp
offset2 = 0.0     # V
command2 = f'APPLy:{waveform2}:CH2 {freq2},{amp2},{offset2}'

phase2 = 80.0       # deg
command2_phase = f'PHAS:CH2 {phase2}'

print("Commands for CH1:", command1, "phase: ", command1_phase)
print("Commands for CH2:", command2, "phase: ", command2_phase)

Commands for CH1: APPLy:SIN 5000000.0,20.0,0.0 phase:  PHAS -70.0
Commands for CH2: APPLy:SIN:CH2 5000000.0,2.0,0.0 phase:  PHAS:CH2 80.0


In [29]:
# Save configuration to file
config_data = {
    'waveform1': waveform1,
    'frequency1': freq1,
    'amplitude1': amp1,
    'offset1': offset1,
    'phase1': phase1,
    'waveform2': waveform2,
    'frequency2': freq2,
    'amplitude2': amp2,
    'offset2': offset2,
    'phase2': phase2
}

save_file_name = 'lock_shg.json'
save_file_path = os.path.join(CONFIG_DIR, save_file_name)
with open(save_file_path, 'w') as f:
    json.dump(config_data, f, indent=4)

### Apply settings to Rigol DG1022

In [ ]:
# CH1
rigol.write(command1)
time.sleep(0.05)  # Short delay to ensure command is processed
rigol.write(command1_phase)
time.sleep(0.05)

In [ ]:
# CH2
rigol.write(command2)
time.sleep(0.05)
rigol.write(command2_phase)
time.sleep(0.05)

### Enable outputs

In [21]:
# Enable CH1 output
rigol.write('OUTP ON')
time.sleep(0.05)

In [22]:
# Enable CH2 output
rigol.write('OUTP:CH2 ON')
time.sleep(0.05)

### Disable outputs

In [23]:
# Disable CH1 output
rigol.write('OUTP OFF')
time.sleep(0.05)

In [24]:
# Disable CH2 output
rigol.write('OUTP:CH2 OFF')
time.sleep(0.05)